---
## 0. Imports

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("Imports OK.")

---
## 1. Configuration

Define your scenes below. Each scene needs:
- `label`: a short name used for filenames and plot titles
- `blue`, `green`, `red`, `nir`: paths to the 4 individual band GeoTIFFs

**Add or remove scenes freely — the rest of the notebook loops over this list.**

In [ ]:
DATA_DIR = "../data"  # Base directory for your band files

SCENES = [
    {
        "label": "scene_20240115",
        "blue":  f"{DATA_DIR}/CBERS_4_WFI_20240115_000_000_L4_BAND13.tif",
        "green": f"{DATA_DIR}/CBERS_4_WFI_20240115_000_000_L4_BAND14.tif",
        "red":   f"{DATA_DIR}/CBERS_4_WFI_20240115_000_000_L4_BAND15.tif",
        "nir":   f"{DATA_DIR}/CBERS_4_WFI_20240115_000_000_L4_BAND16.tif",
    },
    {
        "label": "scene_20240208",
        "blue":  f"{DATA_DIR}/CBERS_4_WFI_20240208_000_000_L4_BAND13.tif",
        "green": f"{DATA_DIR}/CBERS_4_WFI_20240208_000_000_L4_BAND14.tif",
        "red":   f"{DATA_DIR}/CBERS_4_WFI_20240208_000_000_L4_BAND15.tif",
        "nir":   f"{DATA_DIR}/CBERS_4_WFI_20240208_000_000_L4_BAND16.tif",
    },

    # {
    #     "label": "scene_YYYYMMDD",
    #     "blue":  f"{DATA_DIR}/..._BAND13.tif",
    #     "green": f"{DATA_DIR}/..._BAND14.tif",
    #     "red":   f"{DATA_DIR}/..._BAND15.tif",
    #     "nir":   f"{DATA_DIR}/..._BAND16.tif",
    # },
]

# Normalization method
NORM_METHOD = "esun"   # "esun" for CBERS-4, "maxdn" for CBERS-4A/Amazonia-1

# Output directory
OUTPUT_DIR = "../output/cbers4_multi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Scenes to process: {len(SCENES)}")
for i, s in enumerate(SCENES):
    print(f"  [{i}] {s['label']}")

---
## 2. Constants

In [ ]:
CBERS4_WFI_GAIN = {"blue": 0.379, "green": 0.498, "red": 0.360, "nir": 0.351}
ESUN = {"blue": 1984.65, "green": 1823.40, "red": 1559.20, "nir": 1076.30}

HUE_THRESHOLDS = {
    "WATER":   [(16, 35)],
    "WATER95": [(35, 36), (324, 360), (0, 16)],
    "WATER90": [(36, 37), (308, 324)],
    "WATER80": [(37, 160)],
}
CLASS_CODES = {"NON_WATER": 0, "WATER": 1, "WATER95": 2, "WATER90": 3, "WATER80": 4}
MNR_WATER80_MAX = 0.320
MNR_UPPER_CUTOFF = 0.475

print("Constants loaded.")

---
## 3. Processing Functions

In [ ]:
def rgb_to_hsv(r, g, b):
    """Vectorized RGB → HSV (Foley et al., 1996). Returns Hue in [0,360)."""
    cmax = np.maximum(np.maximum(r, g), b)
    cmin = np.minimum(np.minimum(r, g), b)
    delta = cmax - cmin
    h = np.zeros_like(r, dtype=np.float64)
    mask_r = (cmax == r) & (delta > 0)
    h[mask_r] = 60.0 * (((g[mask_r] - b[mask_r]) / delta[mask_r]) % 6.0)
    mask_g = (cmax == g) & (delta > 0) & ~mask_r
    h[mask_g] = 60.0 * (((b[mask_g] - r[mask_g]) / delta[mask_g]) + 2.0)
    mask_b = (cmax == b) & (delta > 0) & ~mask_r & ~mask_g
    h[mask_b] = 60.0 * (((r[mask_b] - g[mask_b]) / delta[mask_b]) + 4.0)
    h = h % 360.0
    s = np.where(cmax > 0, delta / cmax, 0.0)
    v = cmax
    return h, s, v


def process_scene(scene, norm_method, output_dir):
    """
    Full Namikawa pipeline for a single scene.
    Returns the output file path.
    """
    label = scene["label"]
    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # --- Load bands ---
    bands_dn = {}
    profile = None
    for band_name in ["blue", "green", "red", "nir"]:
        with rasterio.open(scene[band_name]) as src:
            bands_dn[band_name] = src.read(1).astype(np.float64)
            if profile is None:
                profile = src.profile.copy()
    print(f"  Loaded {bands_dn['blue'].shape[0]}×{bands_dn['blue'].shape[1]} pixels")

    nodata_mask = (
        (bands_dn["blue"] == 0) & (bands_dn["green"] == 0) &
        (bands_dn["red"] == 0) & (bands_dn["nir"] == 0)
    )

    # --- Normalize ---
    bands_norm = {}
    if norm_method == "esun":
        for name, data in bands_dn.items():
            bands_norm[name] = np.clip((data * CBERS4_WFI_GAIN[name]) / ESUN[name], 0, 1)
    else:
        for name, data in bands_dn.items():
            dmax = np.nanmax(data[~nodata_mask])
            bands_norm[name] = np.clip(data / dmax, 0, 1) if dmax > 0 else np.zeros_like(data)

    # --- HSV (R2G3B5: Green→R, Red→G, NIR→B) ---
    hue, _, _ = rgb_to_hsv(bands_norm["green"], bands_norm["red"], bands_norm["nir"])

    # --- Hue classification ---
    result = np.zeros(hue.shape, dtype=np.uint8)
    for class_name in ["WATER80", "WATER90", "WATER95", "WATER"]:
        code = CLASS_CODES[class_name]
        mask = np.zeros(hue.shape, dtype=bool)
        for hmin, hmax in HUE_THRESHOLDS[class_name]:
            if hmin <= hmax:
                mask |= (hue >= hmin) & (hue < hmax)
            else:
                mask |= (hue >= hmin) | (hue < hmax)
        result[mask] = code
    result[nodata_mask] = 255

    # --- MNR filter ---
    stack = np.stack([bands_norm[b] for b in ["blue", "green", "red", "nir"]], axis=0)
    mnr = np.min(stack, axis=0)
    result[(result == CLASS_CODES["WATER80"]) & (mnr > MNR_WATER80_MAX)] = 0
    result[(result > 0) & (result < 255) & (mnr > MNR_UPPER_CUTOFF)] = 0
    result[nodata_mask] = 255

    # --- Save ---
    out_path = os.path.join(output_dir, f"water_mask_{label}.tif")
    out_profile = profile.copy()
    out_profile.update(dtype=rasterio.uint8, count=1, nodata=255, compress="lzw")
    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(result, 1)

    # --- Stats ---
    n_valid = (~nodata_mask).sum()
    for cname, code in CLASS_CODES.items():
        if code == 0: continue
        n = (result == code).sum()
        print(f"  {cname:10s}: {n:>10,d} px ({100*n/n_valid:.2f}%)")
    total_w = ((result > 0) & (result < 255)).sum()
    print(f"  TOTAL WATER: {total_w:,d} px ({100*total_w/n_valid:.2f}%)")
    print(f"  Saved: {out_path}")

    return out_path, result, nodata_mask, bands_norm

print("Functions defined.")

---
## 4. Process All Scenes

In [ ]:
results = {}  # Store results keyed by label

for scene in SCENES:
    label = scene["label"]
    out_path, mask, nodata, bnorm = process_scene(scene, NORM_METHOD, OUTPUT_DIR)
    results[label] = {
        "path": out_path,
        "mask": mask,
        "nodata": nodata,
        "bands_norm": bnorm,
    }

print(f"\n\nAll {len(results)} scenes processed.")
print("Output files:")
for label, r in results.items():
    print(f"  {label}: {r['path']}")

---
## 5. Visual Summary — All Scenes

In [ ]:
n = len(results)
fig, axes = plt.subplots(2, n, figsize=(6 * n, 10))
if n == 1:
    axes = axes.reshape(2, 1)

cmap = ListedColormap(["#e0e0e0", "#0000cc", "#3366ff", "#6699ff", "#99ccff"])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm_cmap = BoundaryNorm(bounds, cmap.N)

for i, (label, r) in enumerate(results.items()):
    nodata = r["nodata"]
    bn = r["bands_norm"]

    # Top row: false color
    p98 = lambda arr: max(np.percentile(arr[~nodata], 98), 1e-6)
    fc = np.stack([
        np.clip(bn["nir"]   / p98(bn["nir"]),   0, 1),
        np.clip(bn["red"]   / p98(bn["red"]),   0, 1),
        np.clip(bn["green"] / p98(bn["green"]), 0, 1),
    ], axis=-1)
    fc[nodata] = 0
    axes[0, i].imshow(fc)
    axes[0, i].set_title(f"{label}\nFalse Color (NIR-R-G)", fontsize=9)
    axes[0, i].axis("off")

    # Bottom row: water mask
    display = r["mask"].astype(float).copy()
    display[nodata] = np.nan
    im = axes[1, i].imshow(display, cmap=cmap, norm=norm_cmap)
    total_w = ((r["mask"] > 0) & (r["mask"] < 255)).sum()
    axes[1, i].set_title(f"Water Mask\n{total_w:,d} water px", fontsize=9)
    axes[1, i].axis("off")

fig.suptitle("CBERS-4 Water Masks — All Scenes", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "summary_all_scenes.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Done.")